In [87]:
import sys
import os
from pathlib import Path
import time
import itertools

import cv2 as cv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
from joblib import Parallel, delayed
from scipy.spatial import distance_matrix

plt.ioff()

In [17]:
class Sample:
    """Class that contains relevant data of a sample investigated with profilometer"""
    # Conversion factor
    UM2MM = 1e-3
    
    def __init__(self, data_path, time, sample_id):
        self.data_path = data_path
        self.sample_id = sample_id
        
        # Read metadata from CSV
        height_path = data_path / time / "height" / f"{sample_id}.csv"
        with open(height_path, 'r') as file:
            for _ in range(21):
                line = next(file).strip().replace('"', '').split(",")
    
                if line[0] == "XY calibration":
                    # Convert from um to mm
                    self.pixel_size = float(line[1]) * Sample.UM2MM
                if line[0] == "Horizontal":
                    self.horizontal = int(line[1])
                if line[0] == "Vertical":
                    self.vertical = int(line[1])
                if line[0] == "Unit":
                    self.unit = line[1]
        
        # Read height profile of sample from CSV
        height = np.genfromtxt(height_path, skip_header=23, delimiter=",", dtype=str)
        height = np.char.strip(height, '"')
        height[height == ''] = np.nan
        self.height = height.astype(float)

        # Read profilometer image from PNG file
        image_path = data_path / time / "profilometer" / f"{sample_id}.png"
        self.image = cv.imread(image_path, cv.IMREAD_COLOR)

In [18]:
def center_image(image, circle):
    x, y, r = circle
    top_cut = 0 if y - r < 0 else y - r
    left_cut = 0 if x - r < 0 else x - r
    return image[top_cut:y + r + 1, left_cut:x + r + 1], np.array((top_cut, left_cut), dtype=int)


In [19]:
def save_sidebyside(before, after, ident, save_path):
    """Save side by side image to see where the problem lies"""
    if before.ndim == 3:
        cv.imwrite(Path("/data/dust/user/dammal/beegfs.migration/repos/ai2/data/processed/after/sample-masking/") / save_path / f"{ident}.png", np.hstack((before, after)))
    else:
        fig, ax = plt.subplots(1,2)
        ax[0].imshow(np.log(np.abs(before)))
        ax[1].imshow(np.log(np.abs(after)))
        for a in ax:
            a.set_xticklabels([])
            a.set_yticklabels([])
        plt.subplots_adjust(wspace=0, hspace=0)
        plt.savefig(Path("/data/dust/user/dammal/beegfs.migration/repos/ai2/data/processed/after/sample-masking/") / save_path / f"{ident}.png", bbox_inches='tight', dpi=400)
        plt.close()

In [20]:
def save_zebra(before, after, ident, save_path, freq=60):
    """Save side by side image to see where the problem lies"""
    zebra = before.copy()
    
    stripe_width = after.shape[1] // freq
    
    for i in range(freq)[1::2]:
        zebra[:after.shape[0],i*stripe_width:(i+1)*stripe_width] = after[:,i*stripe_width:(i+1)*stripe_width]

    if zebra.ndim == 3:
        cv.imwrite(Path("/data/dust/user/dammal/beegfs.migration/repos/ai2/data/processed/after/sample-masking/") / save_path / f"{ident}.png", zebra)
    else:
        fig, ax = plt.subplots()
        ax.imshow(np.log(np.abs(zebra)))
        # img.set_cmap('')
        ax.axis('off')
        plt.savefig(Path("/data/dust/user/dammal/beegfs.migration/repos/ai2/data/processed/after/sample-masking/") / save_path / f"{ident}.png", bbox_inches='tight', dpi=400)
        plt.close()

In [21]:
def template_match(template, target, shift):
    # Create a gray scale template
    template_gray = cv.cvtColor(template, cv.COLOR_BGR2GRAY)
    # h, w = template.shape

    # Create a gray scale target
    target_gray = cv.cvtColor(target, cv.COLOR_BGR2GRAY)
    
    # 6 methods for template matching
    methods = ['TM_CCOEFF', 'TM_CCOEFF_NORMED', 'TM_CCORR',
                'TM_CCORR_NORMED', 'TM_SQDIFF', 'TM_SQDIFF_NORMED']
    
    top_left_list = []
    
    for meth in methods:
        # Get the method of interest
        method = getattr(cv, meth)
     
        # Apply template Matching
        res = cv.matchTemplate(target_gray, template_gray, method)
        min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)
        
        # If the method is TM_SQDIFF or TM_SQDIFF_NORMED, take minimum
        if method in [cv.TM_SQDIFF, cv.TM_SQDIFF_NORMED]:
            top_left = min_loc
        else:
            top_left = max_loc

        top_left = top_left - np.array(shift[::-1])

        # Store the top left position in target that matches with template
        top_left_list.append(top_left)

    # Determine the top left position proposals of the different methods and count agreement
    return top_left_list

In [22]:
def equalize_size(centered_before, centered_after):
    if centered_before.shape == centered_after.shape:
        return centered_before, centered_after
    else:
        min1 = np.min((centered_before.shape[0], centered_after.shape[0]))
        min2 = np.min((centered_before.shape[1], centered_after.shape[1]))
        return centered_before[:min1, :min2], centered_after[:min1, :min2]

In [23]:
def overlay_images(top_left, image):
    """Pad the after image if necessary or cut away the left and top row to overlay the after image with the before image"""
    if top_left[1] < 0:
        if image.ndim == 3:
            image = np.pad(image, ((-top_left[1], 0), (0, 0), (0, 0)))
        else:
            image = np.pad(image, ((-top_left[1], 0), (0, 0)))
    elif top_left[1] > 0:
        image = image[top_left[1]:,:]
    else:
        image = image

    if top_left[0] < 0:
        if image.ndim == 3:
            image = np.pad(image, ((0, 0), (-top_left[0], 0), (0, 0)))
        else:
            image = np.pad(image, ((0, 0), (-top_left[0], 0)))
    elif top_left[0] > 0:
        image = image[:,top_left[0]:]
    else:
        image = image

    return image


In [24]:
def match_samples_statistically(before, after, cut_top=10, cut_left=10, cut_bottom=5, cut_right=5, freq=10):
    top_left_list = []

    reduced_before = before[cut_top:-(cut_bottom-1), cut_left:-(cut_right-1)]
    step = np.array(reduced_before.shape) // (freq + 1)
    
    for i in range(freq-1):
        for ii in range(freq-1):
            template = reduced_before[i*step[0]:(i+1)*step[0],ii*step[1]:(ii+1)*step[1]]            
            top_left_list += template_match(template, after, (cut_top + i*step[0], cut_left + ii*step[1]))

    template = reduced_before[freq*step[0]:,freq*step[1]:]
    top_left_list += template_match(template, after, ((cut_top + freq*step[0], cut_left + freq*step[1])))
    
    top_left, certainty = determine_optimal_top_left(top_left_list)
        
    return top_left, certainty
    

In [133]:
def calc_smallest_distance(position_array):
    return position_array[np.argmin(np.mean(distance_matrix(np.expand_dims(position_array, axis=1), np.expand_dims(position_array, axis=1), p=1), axis=1))]

In [139]:
def match_sample_borders(before, after, cut_top=10, cut_left=10, cut_bottom=5, cut_right=5, freq=10, filter_range=100):
    top_left_list = []

    reduced_before = before[cut_top:-(cut_bottom-1), cut_left:-(cut_right-1)]
    
    # Determine template matches for 4 subpatches of the centered before image
    percentage_list = ([0.0, 0.1], [0.9, 1.0])
    percentage_combination = list(itertools.product(percentage_list, percentage_list))
    for percentage in percentage_combination:
        before_patch, top_left_shift = get_patch(reduced_before, *percentage)
        top_left_list += template_match(before_patch, after, np.array((cut_top, cut_left)) + top_left_shift)

    top_left_list = np.array(top_left_list)
    # Filter for image offsets smaller than filter_range pixel
    top_left_list = top_left_list[np.all(np.abs(top_left_list) <= filter_range, axis=1)]
    
    # top_left = top_left_list[np.argmin(np.mean(distance_matrix(top_left_list, top_left_list), axis=1))]
    return (calc_smallest_distance(top_left_list[:, 0]), calc_smallest_distance(top_left_list[:, 1]))
    

In [25]:
def determine_optimal_top_left(top_left_list):
    coord1, counts1 = np.unique(np.array(top_left_list)[:, 0], axis=0, return_counts=True)
    counts1_index = np.argmax(counts1)

    coord2, counts2 = np.unique(np.array(top_left_list)[:, 1], axis=0, return_counts=True)
    counts2_index = np.argmax(counts2)

    top_left = (coord1[counts1_index], coord2[counts2_index])
    certainty = (counts1[counts1_index] / np.sum(counts1), counts2[counts2_index] / np.sum(counts2))

    return top_left, certainty

In [26]:
def get_patch(image, top_left_percentage, bottom_right_percentage):
    shape = np.array(image.shape)
    y_cut = np.rint(np.array(shape)[0] * np.array(top_left_percentage)).astype(int)
    x_cut = np.rint(np.array(shape)[1] * np.array(bottom_right_percentage)).astype(int)
    return image[y_cut[0]:y_cut[1], x_cut[0]:x_cut[1]], np.array((y_cut[0], x_cut[0]))

In [27]:
def match_samples_directly(before, after, circle, distance_limit=10):
    top_left_patch_list = []

    # Determine best fit with centered image
    centered_before, cut_coords = center_image(before, circle)
    top_left_list = template_match(centered_before, after, cut_coords)
    top_left, certainty = determine_optimal_top_left(top_left_list)

    # Determine template matches for 5 subpatches of the centered before image
    percentage_list = ([0.05, 0.45], [0.55, 0.95])
    percentage_combination = list(itertools.product(percentage_list, percentage_list))
    percentage_combination.append(([0.3, 0.7], [0.3, 0.7]))
    for percentage in percentage_combination:
        centered_before_patch, top_left_shift = get_patch(centered_before, *percentage)
        top_left_list = template_match(centered_before_patch, after, cut_coords + top_left_shift)
        top_left_patch, patch_certainty = determine_optimal_top_left(top_left_list)
        top_left_patch_list.append(top_left_patch)

    # Determine precision and consistency of subpatches with large matched patch
    radial_distance = np.linalg.norm((np.array(top_left_patch_list) - top_left), axis=1)
    valid_coords = radial_distance[radial_distance <= distance_limit]
    precision = np.mean(valid_coords)
    consistency = len(valid_coords)
    
    return top_left, consistency, precision
    

In [137]:
def center_samples(path, ident, circle_df, consistency_limit=0):
    certainty = None
    
    # Determine the centering circle for image
    circle = circle_df.loc[[ident]].values[0]

    # Load before and after images and heights
    try:
        after = Sample(path, "after", ident)
    except FileNotFoundError:
        return
    
    before = Sample(path, "before", ident)

    before_image = before.image
    after_image = after.image

    before_height = before.height
    after_height = after.height

    # Determine positional overlay of before and after image by template matching the centered before image to after
    top_left, consistency, precision = match_samples_directly(before_image, after_image, circle)

    # If consistency in first template matching was to low, use statistical template matching procedure
    if consistency <= consistency_limit:
        # top_left, certainty = match_samples_statistically(before_image, after_image)
        top_left = match_sample_borders(before_image, after_image)

    # Overlay the after image and height with before images and height
    after_image = overlay_images(top_left, after_image)
    after_height = overlay_images(top_left, after_height)

    # Make overlayed arrays equal in size
    before_image, after_image = equalize_size(before_image, after_image)
    before_height, after_height = equalize_size(before_height, after_height)

    # Save side by side images
    save_sidebyside(before_image, after_image, ident, "side-by-side-image")
    save_sidebyside(before_height, after_height, ident, "side-by-side-height")

    # Save zebra images
    save_zebra(before_image, after_image, ident, "zebra-image")
    save_zebra(before_height, after_height, ident, "zebra-height")

    if consistency <= 1:
        save_sidebyside(before_image, after_image, ident, "side-by-side-image-low-consistency")
        save_zebra(before_image, after_image, ident, "zebra-image-low-consistency")
        
    # Center arrays
    centered_before_image, _ = center_image(before_image, circle)
    centered_after_image, _ = center_image(after_image, circle)
    centered_before_height, _ = center_image(before_height, circle)
    centered_after_height, _ = center_image(after_height, circle)
    
    # Save centered images
    cv.imwrite(path / "before/centered-image" / f"{ident}.png", centered_before_image)
    cv.imwrite(path / "after/centered-image" / f"{ident}.png", centered_after_image)
    # Save centered height maps
    np.save(path / "before/centered-height" / f"{ident}.npy", centered_before_height)
    np.save(path / "after/centered-height" / f"{ident}.npy", centered_after_height)

    return ident, consistency

In [148]:
def center_samples_wrapper(path, ident, circle_df):
    try:
        return center_samples(path, ident, circle_df)
    except Exception as e:
        print(ident)
        return ident, -1

In [141]:
path = Path("/data/dust/user/dammal/beegfs.migration/repos/ai2/data/processed")
time = "before"
sample_ids = [i.replace(".png", "") for i in os.listdir(path / time / "profilometer")]
ncpu = os.cpu_count()
circle_df = pd.read_csv(path / time / "sample-masking/circle-list/circle-list.csv", index_col = 0)

In [149]:
result = Parallel(n_jobs=ncpu)(delayed(center_samples_wrapper)(path, ident, circle_df) for ident in sample_ids)

2-1-10-1


In [ ]:
# result = [center_samples_wrapper(path, ident, circle_df) for ident in sample_ids]

In [151]:
result

[('4-4-1-3', 3),
 ('2-5-2-1', 5),
 ('1-1-4-4', 5),
 ('2-1-12-3', 4),
 ('4-6-1-4', 4),
 ('1-1-9-3', 5),
 ('1-6-2-1', 5),
 ('1-5-10-3', 3),
 ('1-1-4-2', 5),
 ('2-5-11-3', 5),
 ('1-2-2-4', 5),
 ('4-1-2-1', 1),
 ('4-6-1-1', 1),
 ('2-4-11-3', 5),
 ('4-6-2-1', 4),
 ('1-2-3-3', 5),
 ('1-5-6-1', 5),
 ('2-4-11-4', 5),
 ('1-3-3-2', 5),
 ('1-5-4-1', 5),
 ('1-2-9-3', 5),
 ('3-5-12-4', 5),
 ('1-3-9-2', 5),
 ('2-1-1-1', 4),
 ('4-3-1-1', 5),
 ('3-7-7-1', 2),
 ('3-7-8-1', 5),
 ('2-4-6-1', 5),
 ('4-1-1-4', 5),
 ('2-1-2-4', 5),
 ('3-8-4-4', 5),
 ('2-2-2-1', 5),
 ('1-3-4-1', 5),
 ('2-2-2-3', 5),
 ('4-5-2-4', 5),
 ('4-7-1-3', 0),
 ('1-5-8-3', 5),
 ('4-4-8-3', 0),
 ('2-1-2-3', 5),
 ('3-1-2-4', 5),
 None,
 ('3-8-1-4', 2),
 ('2-1-1-3', 4),
 ('2-4-9-1', 5),
 ('4-7-12-4', 0),
 ('1-6-7-1', 5),
 ('4-6-5-4', 5),
 ('2-1-8-3', 3),
 ('1-1-5-4', 5),
 ('1-5-4-4', 5),
 ('1-3-2-1', 5),
 ('4-3-6-3', 5),
 ('4-5-3-1', 5),
 ('1-5-9-1', 5),
 ('2-3-3-2', 4),
 ('4-5-8-3', 5),
 ('4-5-2-3', 5),
 ('2-3-6-1', 5),
 ('1-3-7-1', 5),


In [152]:
ident = np.array([i[0] for i in result if i is not None])
consistency = np.array([i[1] for i in result if i is not None])

indices = np.argsort(consistency)
ident, consistency[indices]

(array(['4-4-1-3', '2-5-2-1', '1-1-4-4', '2-1-12-3', '4-6-1-4', '1-1-9-3',
        '1-6-2-1', '1-5-10-3', '1-1-4-2', '2-5-11-3', '1-2-2-4', '4-1-2-1',
        '4-6-1-1', '2-4-11-3', '4-6-2-1', '1-2-3-3', '1-5-6-1', '2-4-11-4',
        '1-3-3-2', '1-5-4-1', '1-2-9-3', '3-5-12-4', '1-3-9-2', '2-1-1-1',
        '4-3-1-1', '3-7-7-1', '3-7-8-1', '2-4-6-1', '4-1-1-4', '2-1-2-4',
        '3-8-4-4', '2-2-2-1', '1-3-4-1', '2-2-2-3', '4-5-2-4', '4-7-1-3',
        '1-5-8-3', '4-4-8-3', '2-1-2-3', '3-1-2-4', '3-8-1-4', '2-1-1-3',
        '2-4-9-1', '4-7-12-4', '1-6-7-1', '4-6-5-4', '2-1-8-3', '1-1-5-4',
        '1-5-4-4', '1-3-2-1', '4-3-6-3', '4-5-3-1', '1-5-9-1', '2-3-3-2',
        '4-5-8-3', '4-5-2-3', '2-3-6-1', '1-3-7-1', '3-7-1-1', '1-1-2-1',
        '4-3-3-1', '1-3-3-3', '4-6-3-3', '1-1-1-1', '4-7-3-2', '3-1-4-4',
        '3-5-3-4', '4-5-6-4', '4-4-9-1', '1-5-2-4', '1-2-6-2', '1-6-3-2',
        '3-8-5-3', '1-3-9-3', '1-2-4-1', '3-9-5-3', '4-5-3-4', '1-1-11-3',
        '1-1-2-4', '1-3-8-3', 

In [153]:
center_samples(path, '2-1-10-1',circle_df)

ValueError: attempt to get argmin of an empty sequence

In [46]:
ident = '4-4-9-3'
certainty = None

# Determine the centering circle for image
circle = circle_df.loc[[ident]].values[0]

# Load before and after images and heights
after = Sample(path, "after", ident)

before = Sample(path, "before", ident)

before_image = before.image
after_image = after.image

before_height = before.height
after_height = after.height

# # Determine positional overlay of before and after image by template matching the centered before image to after
# top_left, consistency, precision = match_samples_directly(before_image, after_image, circle)

# # If consistency in first template matching was to low, use statistical template matching procedure
# if consistency <= consistency_limit:
#     top_left, certainty = match_samples_statistically(before_image, after_image)

# # Overlay the after image and height with before images and height
# after_image = overlay_images(top_left, after_image)
# after_height = overlay_images(top_left, after_height)

# # Make overlayed arrays equal in size
# before_image, after_image = equalize_size(before_image, after_image)
# before_height, after_height = equalize_size(before_height, after_height)

# # Save side by side images
# save_sidebyside(before_image, after_image, ident, "side-by-side-image")
# save_sidebyside(before_height, after_height, ident, "side-by-side-height")

# # Save zebra images
# save_zebra(before_image, after_image, ident, "zebra-image")
# save_zebra(before_height, after_height, ident, "zebra-height")

# if consistency <= 1:
#     save_sidebyside(before_image, after_image, ident, "side-by-side-image-low-consistency")
#     save_zebra(before_image, after_image, ident, "zebra-image-low-consistency")
    
# # Center arrays
# centered_before_image, _ = center_image(before_image, circle)
# centered_after_image, _ = center_image(after_image, circle)
# centered_before_height, _ = center_image(before_height, circle)
# centered_after_height, _ = center_image(after_height, circle)

# # Save centered images
# cv.imwrite(path / "before/centered-image" / f"{ident}.png", centered_before_image)
# cv.imwrite(path / "after/centered-image" / f"{ident}.png", centered_after_image)
# # Save centered height maps
# np.save(path / "before/centered-height" / f"{ident}.npy", centered_before_height)
# np.save(path / "after/centered-height" / f"{ident}.npy", centered_after_height)

# return ident, consistency, certainty